# 05 · Build the final test set (balanced: examiner and applicant citations)

Replaces the first test set, which only kept cases with at least 2 **examiner** citations and so under-represented **applicant** citations.

**New rule (no preference for either side):** every recent G06N patent family (priority date 2022 or later, outside the corpus) whose publications cite **at least 2 corpus patents of any kind**. All such families are kept, no sampling: extracting more text costs the same quota, because BigQuery charges by whole columns.

**Query = the first independent claim as filed** (the published application). That is the claim the examiner searched; a granted claim has already been narrowed away from the cited prior art.

One BigQuery query reads only **claims, title and applicant** (about 130 GB, capped at 200 GB). Saves `testset.parquet` and `qrels.parquet` to Drive, replacing the earlier versions.

In [ ]:
BRANCH = "v2-corpus"
!rm -rf /content/app
!git clone -q --depth 1 -b {BRANCH} https://github.com/Renuk9390/patent-prior-art-explorer.git /content/app
import sys; sys.path.insert(0, "/content/app/src")

PROJECT_ID = "patent-search-515813"
TABLE = "patents-public-data.patents.publications"
DRIVE_DIR = "/content/drive/MyDrive/patent-search"

from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

import pandas as pd
from google.cloud import bigquery
from patentsearch.eval.qrels import build_qrels, citation_source
from patentsearch.ingest.patents import first_independent_claim
client = bigquery.Client(project=PROJECT_ID)

meta = pd.read_parquet(f"{DRIVE_DIR}/g06n_us_meta.parquet")
cit = pd.read_parquet(f"{DRIVE_DIR}/g06n_us_citations.parquet", columns=["publication_number", "cited", "category"])
patents = pd.read_parquet(f"{DRIVE_DIR}/patents.parquet")
print(f"{len(meta):,} G06N documents, {len(cit):,} citations, {len(patents):,} corpus patents")

## 1 · Candidate test families

In [ ]:
family_of = dict(zip(meta["publication_number"], meta["family_id"]))
corpus_doc_of_family = dict(zip(patents["family_id"], patents["publication_number"]))

# Recent families outside the corpus: every publication with a 2022+ priority date
recent = meta[(meta["priority_date"] >= 20220101) & ~meta["family_id"].isin(corpus_doc_of_family)]
recent_families = set(recent["family_id"])

c = cit.assign(citing_family=cit["publication_number"].map(family_of),
               cited_family=cit["cited"].map(family_of))
c = c[c["citing_family"].isin(recent_families) & c["cited_family"].isin(corpus_doc_of_family)]
labels = c.groupby(["citing_family", "cited_family"])["category"].apply(citation_source).rename("source").reset_index()
labels = labels[labels["source"] != "other"]

n_cited = labels.groupby("citing_family")["cited_family"].nunique()
families = set(n_cited[n_cited >= 2].index)
lab = labels[labels["citing_family"].isin(families)]
per_f = lab.groupby("citing_family")["source"].agg(set)

print(f"Candidate families (>=2 corpus citations of any kind): {len(families):,}")
print("Labels by who cited:")
print(lab["source"].value_counts().to_string())
print(f"Families with >=1 examiner-only citation:  {per_f.apply(lambda s: 'examiner' in s).sum():,}")
print(f"Families with >=1 applicant-only citation: {per_f.apply(lambda s: 'applicant' in s).sum():,}")
print(f"Families with both kinds:                  {per_f.apply(lambda s: {'examiner', 'applicant'} <= s).sum():,}")

## 2 · One query document per family: the application as filed

In [ ]:
docs = recent[recent["family_id"].isin(families)].copy()
docs["is_application"] = docs["kind_code"].str.startswith("A")
# Prefer the earliest published application; fall back to the earliest grant
docs = docs.sort_values(["family_id", "is_application", "publication_date"], ascending=[True, False, True])
query_docs = docs.drop_duplicates("family_id", keep="first")
query_of_family = dict(zip(query_docs["family_id"], query_docs["publication_number"]))
print(f"Query documents: {len(query_docs):,} "
      f"({query_docs['is_application'].mean():.0%} applications as filed, rest granted patents)")

## 3 · Extract claims, title and applicant (one query)

In [ ]:
ids = sorted(query_of_family.values())
SQL = f"""
SELECT
  publication_number,
  (SELECT t.text FROM UNNEST(title_localized) t WHERE t.language = 'en' LIMIT 1) AS title,
  (SELECT t.text FROM UNNEST(claims_localized) t WHERE t.language = 'en' LIMIT 1) AS claims,
  ARRAY(SELECT a.name FROM UNNEST(assignee_harmonized) a) AS assignees
FROM `{TABLE}`
WHERE publication_number IN UNNEST(@ids)
"""
params = [bigquery.ArrayQueryParameter("ids", "STRING", ids)]
dry = client.query(SQL, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False, query_parameters=params))
gb = dry.total_bytes_processed / 1e9
print(f"Documents: {len(ids):,} | estimated scan: {gb:.1f} GB")

MAX_GB = 200
assert gb <= MAX_GB, "Above the limit: stopping. Nothing was charged."
text = client.query(SQL, job_config=bigquery.QueryJobConfig(
    query_parameters=params, maximum_bytes_billed=int(MAX_GB * 1e9))).to_dataframe()
text["assignees"] = text["assignees"].apply(lambda a: "; ".join(a) if a is not None else "")
print(f"Extracted {len(text):,} documents")

## 4 · Final test set and relevance labels

In [ ]:
testset = query_docs[["publication_number", "family_id", "kind_code", "priority_date"]].merge(text, on="publication_number")
testset["query"] = testset["claims"].apply(first_independent_claim)
dropped = testset["query"].isna().sum()
testset = testset.dropna(subset=["query"]).drop(columns=["claims"])
testset["url"] = "https://patents.google.com/patent/" + testset["publication_number"].str.replace("-", "")
print(f"Test cases with a usable first claim: {len(testset):,} (dropped {dropped})")
print("Query length in words: median", int(testset["query"].str.split().str.len().median()))

kept = dict((f, q) for f, q in query_of_family.items() if q in set(testset["publication_number"]))
qrels = build_qrels(cit, family_of, kept, corpus_doc_of_family,
                    dict(zip(testset["publication_number"], testset["assignees"])),
                    dict(zip(patents["publication_number"], patents["assignees"])))
qrels = qrels[qrels["source"] != "other"]

print(f"\nRelevance labels: {len(qrels):,} ({len(qrels) / qrels.query_id.nunique():.1f} per test case)")
print(qrels["source"].value_counts().to_string())
per_q = qrels.groupby("query_id")["source"].agg(set)
print(f"Test cases with >=1 examiner-only citation:  {per_q.apply(lambda s: 'examiner' in s).sum():,}")
print(f"Test cases with >=1 applicant-only citation: {per_q.apply(lambda s: 'applicant' in s).sum():,}")
print(f"Test cases with both kinds (paired):         {per_q.apply(lambda s: {'examiner', 'applicant'} <= s).sum():,}")
print("\nSame applicant as the test case, by who cited:")
print(qrels.groupby("source")["same_applicant"].mean().round(3).to_string())

In [ ]:
testset.to_parquet(f"{DRIVE_DIR}/testset.parquet", index=False)
qrels.to_parquet(f"{DRIVE_DIR}/qrels.parquet", index=False)
q = testset.iloc[0]
print(f"Example {q.publication_number} ({q.kind_code}): {q.title} | {q.assignees}")
print("Query:", q.query[:250], "...")
print("Cited corpus patents:", qrels[qrels.query_id == q.publication_number][["doc_id", "source"]].values.tolist())